# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates step-by-step exploration of the [FAIR^2](https://sen.science/doi/10.71728/senscience.qs2f-h81p) dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) Python library.

### Dataset Source
The dataset source is provided via a [Croissant](https://mlcommons.org/croissant/) schema URL.

In [ ]:
# Ensure `mlcroissant` and core visualization packages are installed
!pip install -U mlcroissant matplotlib seaborn

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset Croissant metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}\n\nDescription: {metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs (`@id`).

> All entities, including record sets and fields, are referenced by their `@id` as required in FAIR workflows.

In [ ]:
# Let's enumerate all available record sets (by their @id) and their fields.
record_sets = list(dataset.record_sets)
if len(record_sets) == 0:
    print("No record sets are directly linked at the root level. Scanning attached distributions for tabular data...")
    # For tabular datasets, we use dataset.tabular_record_sets (if present)
    tabular_set_ids = [rs['@id'] for rs in getattr(dataset, 'tabular_record_sets', [])]
    if len(tabular_set_ids) > 0:
        print(f"Found tabular record sets: {tabular_set_ids}")
        # Display columns/fields in the first tabular record set
        for record_set_id in tabular_set_ids:
            record_set = dataset.get_record_set(record_set_id)
            field_ids = [field['@id'] for field in getattr(record_set, 'fields', [])]
            print(f"\nRecord set {record_set_id} fields: {field_ids}")
    else:
        print("No tabular record sets located. Please check schema structure.")
else:
    record_set_ids = [rs['@id'] for rs in record_sets]
    print(f"Root-level record sets: {record_set_ids}")
    for record_set in record_sets:
        curr = dataset.get_record_set(record_set['@id'])
        if not hasattr(curr, 'fields'):
            continue
        flds = getattr(curr, 'fields', [])
        field_ids = [f['@id'] for f in flds]
        print(f"\nRecord set {record_set['@id']} fields: {field_ids}")

## 3. Data Extraction
Load data from the main record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview step above.

In [ ]:
# Pull the set of available record sets again via tabular_record_sets attribute (most clinical tabular datasets use this).
if hasattr(dataset, 'tabular_record_sets') and len(dataset.tabular_record_sets) > 0:
    record_sets = [rs['@id'] for rs in dataset.tabular_record_sets]
else:
    record_sets = [rs['@id'] for rs in dataset.record_sets]

print('Record sets (@id):', record_sets)

dataframes = {}
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Pick the main clinical record set for display (take the first set if unsure):
main_record_set_id = record_sets[0] if len(record_sets) > 0 else None
if main_record_set_id:
    print(f"\nColumns of {main_record_set_id}:", dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No record sets with data found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. 
All fields/columns are referenced by their `@id`. 

Typical EDA steps include removing outliers, transforming distributions, and grouping data by key variables.

In [ ]:
# Identify a numeric field by inspecting the columns (using @id)
df = dataframes[main_record_set_id]
print("First 5 rows of the main record set:")
display(df.head())

# Try to auto-select a likely numeric field (age, interval, etc) by dtype
numeric_candidates = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
print("Numeric candidate fields (@id):", numeric_candidates)

if len(numeric_candidates) == 0:
    raise Exception("No numeric fields detected. Please check field schema.")

numeric_field_id = numeric_candidates[0]  # Proceed with the first one

# Example filtering: select all records with the numeric field > threshold
threshold = df[numeric_field_id].median()
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold} (using @id):")
display(filtered_df.head())

# Normalize the numeric field (z-scoring)
filtered_df[f"{numeric_field_id}_normalized"] = (
    (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
)

print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by a categorical field (e.g., MSI status, anatomical site)
categorical_candidates = [c for c in df.columns if pd.api.types.is_object_dtype(df[c])]
print("Categorical candidate fields (@id):", categorical_candidates)

group_field_id = None
for cand in categorical_candidates:
    n_uniq = df[cand].nunique()
    if 2 <= n_uniq < 10:
        group_field_id = cand
        break
if group_field_id is not None:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
    display(grouped_df)
else:
    print("No suitable categorical @id found for grouping.")

## 5. Visualization
Visualize the distribution of the selected numeric field and its relationship to the chosen categorical field (by `@id`, if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution plot of the numeric field
plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field_id], kde=True, bins=10)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# If grouping field available, plot group means
if group_field_id is not None:
    plt.figure(figsize=(8,5))
    sns.barplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.ylabel(f"Mean of {numeric_field_id}")
    plt.xlabel(f"{group_field_id}")
    plt.xticks(rotation=30)
    plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the FAIR^2 colorectal cancer clinical dataset via its Croissant schema using the `mlcroissant` library. Data processing used record set and field references by `@id` to ensure schema-driven, reproducible workflows. We visualized field distributions and demonstrated basic grouping/normalization techniques, forming a foundation for further statistical or machine learning analyses.